# **00 — Data Ingestion & Data Understanding**

## Project Overview
**Retail Customer Intelligence Platform** - pipeline end-to-end phân khúc khách hàng: PostgreSQL
(lưu trữ/làm sạch) → Python (RFM + KMeans) → Power BI (dashboard). Notebook này là bước đầu tiên:
hiểu dữ liệu trước khi đụng vào bất kỳ dòng code phân tích nào.

## Dataset Overview
**Olist Brazilian E-Commerce Dataset** - dữ liệu thật từ sàn thương mại điện tử Olist (Brazil),
giai đoạn 2016–2018. Gồm ~100,000 đơn hàng trải dài các khía cạnh: đơn hàng, thanh toán, sản phẩm,
người bán, khách hàng, đánh giá - trải trên 9 bảng CSV có quan hệ với nhau.

## Dataset Source
https://www.kaggle.com/datasets/olistbr/brazilian-ecommerce

## Business Context
Xem chi tiết ở `docs/business_problem.md`. Tóm tắt: doanh nghiệp e-commerce muốn biết khách hàng
nào đáng giữ chân nhất, khách hàng nào có nguy cơ rời bỏ, để phân bổ ngân sách marketing hiệu quả
hơn thay vì đối xử như nhau với toàn bộ ~99,000 khách hàng.

## Mục đích notebook này
1. Giới thiệu 9 bảng dữ liệu và mối quan hệ giữa chúng (ER diagram)
2. Kiểm tra data quality: kích thước, dtype, missing, duplicate, PK/FK, cardinality, date range
3. Ghi lại các quyết định thiết kế schema (đã áp dụng vào `database/schema.sql`)
4. Chạy ingest pipeline và verify

**Lưu ý:** notebook là nơi *khám phá và quyết định*. Logic tái sử dụng nằm ở `src/ingest.py`, `src/database.py`.

## **ER Diagram (mô tả quan hệ giữa 9 bảng)**

```
customers ──1:N── orders ──1:N── order_items ──N:1── products ──N:1── category_translation
                     │                  │
                     │                  └──N:1── sellers
                     │
                     ├──1:N── order_payments
                     └──1:N── order_reviews

geolocation (đứng độc lập, join qua zip_code_prefix — optional, không dùng cho RFM)
```

**Điểm quan trọng về identity:** `customers.customer_id` là ID theo TỪNG ĐƠN HÀNG, không phải
theo từng người. `customers.customer_unique_id` mới là identity thật của 1 khách hàng —
**mọi phép tính RFM/segmentation đều phải group theo `customer_unique_id`**, không phải `customer_id`.

In [1]:
import sys
sys.path.append('..')

import pandas as pd
from src.config import RAW_DATA_DIR

pd.set_option('display.max_columns', None)

## **1. Load 9 bảng CSV gốc**

In [2]:
customers = pd.read_csv(RAW_DATA_DIR / 'olist_customers_dataset.csv')
geolocation = pd.read_csv(RAW_DATA_DIR / 'olist_geolocation_dataset.csv')
order_items = pd.read_csv(RAW_DATA_DIR / 'olist_order_items_dataset.csv')
order_payments = pd.read_csv(RAW_DATA_DIR / 'olist_order_payments_dataset.csv')
order_reviews = pd.read_csv(RAW_DATA_DIR / 'olist_order_reviews_dataset.csv')
orders = pd.read_csv(RAW_DATA_DIR / 'olist_orders_dataset.csv', parse_dates=[
    'order_purchase_timestamp', 'order_approved_at',
    'order_delivered_carrier_date', 'order_delivered_customer_date',
    'order_estimated_delivery_date',
])
products = pd.read_csv(RAW_DATA_DIR / 'olist_products_dataset.csv')
sellers = pd.read_csv(RAW_DATA_DIR / 'olist_sellers_dataset.csv')
category_translation = pd.read_csv(RAW_DATA_DIR / 'product_category_name_translation.csv')

tables = {
    'customers': customers, 'geolocation': geolocation, 'order_items': order_items,
    'order_payments': order_payments, 'order_reviews': order_reviews, 'orders': orders,
    'products': products, 'sellers': sellers, 'category_translation': category_translation,
}

## **2. Shape & Data Types**

In [3]:
for name, df in tables.items():
    print(f'{name:25s} shape={df.shape}')

customers                 shape=(99441, 5)
geolocation               shape=(1000163, 5)
order_items               shape=(112650, 7)
order_payments            shape=(103886, 5)
order_reviews             shape=(99224, 7)
orders                    shape=(99441, 8)
products                  shape=(32951, 9)
sellers                   shape=(3095, 4)
category_translation      shape=(71, 2)


In [4]:
orders.dtypes

order_id                                 object
customer_id                              object
order_status                             object
order_purchase_timestamp         datetime64[ns]
order_approved_at                datetime64[ns]
order_delivered_carrier_date     datetime64[ns]
order_delivered_customer_date    datetime64[ns]
order_estimated_delivery_date    datetime64[ns]
dtype: object

In [5]:
order_items.dtypes

order_id                object
order_item_id            int64
product_id              object
seller_id               object
shipping_limit_date     object
price                  float64
freight_value          float64
dtype: object

## **3. Missing Values**

In [6]:
for name, df in tables.items():
    nulls = df.isnull().sum()
    nulls = nulls[nulls > 0]
    if len(nulls):
        print(f'--- {name} ---')
        print(nulls)
        print()

--- order_reviews ---
review_comment_title      87656
review_comment_message    58247
dtype: int64

--- orders ---
order_approved_at                 160
order_delivered_carrier_date     1783
order_delivered_customer_date    2965
dtype: int64

--- products ---
product_category_name         610
product_name_lenght           610
product_description_lenght    610
product_photos_qty            610
product_weight_g                2
product_length_cm               2
product_height_cm               2
product_width_cm                2
dtype: int64



**Nhận xét:** null trong `orders` (approved_at, delivered_carrier_date, delivered_customer_date)
tăng dần theo tiến trình xử lý đơn — logic nghiệp vụ bình thường (đơn chưa giao thì chưa có ngày
giao), không phải lỗi data. `products.product_category_name` thiếu 610 dòng — giữ nguyên là
NULL/'unknown' khi phân tích theo category thay vì loại bỏ sản phẩm.

## **4. Duplicate Check**

In [7]:
for name, df in tables.items():
    dup = df.duplicated().sum()
    print(f'{name:25s} full-row duplicates: {dup}')

customers                 full-row duplicates: 0
geolocation               full-row duplicates: 261831
order_items               full-row duplicates: 0
order_payments            full-row duplicates: 0
order_reviews             full-row duplicates: 0
orders                    full-row duplicates: 0
products                  full-row duplicates: 0
sellers                   full-row duplicates: 0
category_translation      full-row duplicates: 0


## **5. Primary Key Check**

In [8]:
pk_checks = {
    'customers': 'customer_id',
    'orders': 'order_id',
    'products': 'product_id',
    'sellers': 'seller_id',
}

for table_name, pk_col in pk_checks.items():
    df = tables[table_name]
    n_unique = df[pk_col].nunique()
    n_total = len(df)
    status = 'OK - unique' if n_unique == n_total else 'CANH BAO - trung lap'
    print(f'{table_name:15s} {pk_col:15s} unique={n_unique} total={n_total}  -> {status}')

customers       customer_id     unique=99441 total=99441  -> OK - unique
orders          order_id        unique=99441 total=99441  -> OK - unique
products        product_id      unique=32951 total=32951  -> OK - unique
sellers         seller_id       unique=3095 total=3095  -> OK - unique


In [9]:
# order_reviews.review_id KHONG unique mot minh — finding thuc te da phat hien
print('Trung review_id (mot minh):', order_reviews['review_id'].duplicated().sum())
print('Trung ca (review_id + order_id):', order_reviews.duplicated(subset=['review_id', 'order_id']).sum())
print()
print('=> Ket luan: dung composite key (review_id, order_id) lam PRIMARY KEY, khong dung review_id rieng le.')

Trung review_id (mot minh): 814
Trung ca (review_id + order_id): 0

=> Ket luan: dung composite key (review_id, order_id) lam PRIMARY KEY, khong dung review_id rieng le.


## **6. Foreign Key Check**
Kiểm tra xem có giá trị FK nào ở bảng con không tồn tại ở bảng cha không — nếu có, khi JOIN sẽ mất dữ liệu.

In [10]:
checks = [
    ('orders.customer_id -> customers.customer_id',
     orders['customer_id'], customers['customer_id']),
    ('order_items.order_id -> orders.order_id',
     order_items['order_id'], orders['order_id']),
    ('order_items.product_id -> products.product_id',
     order_items['product_id'], products['product_id']),
    ('order_items.seller_id -> sellers.seller_id',
     order_items['seller_id'], sellers['seller_id']),
    ('order_payments.order_id -> orders.order_id',
     order_payments['order_id'], orders['order_id']),
    ('order_reviews.order_id -> orders.order_id',
     order_reviews['order_id'], orders['order_id']),
]

for label, child_col, parent_col in checks:
    orphans = set(child_col.dropna().unique()) - set(parent_col.unique())
    status = 'OK' if len(orphans) == 0 else f'{len(orphans)} gia tri MO COI'
    print(f'{label:50s} -> {status}')

orders.customer_id -> customers.customer_id        -> OK
order_items.order_id -> orders.order_id            -> OK
order_items.product_id -> products.product_id      -> OK
order_items.seller_id -> sellers.seller_id         -> OK
order_payments.order_id -> orders.order_id         -> OK
order_reviews.order_id -> orders.order_id          -> OK


In [11]:
# Finding thuc te da phat hien: category thieu ban dich
cats_products = set(products['product_category_name'].dropna().unique())
cats_translation = set(category_translation['product_category_name'].unique())
missing_cats = cats_products - cats_translation

print(f'So category trong products: {len(cats_products)}')
print(f'So category co ban dich: {len(cats_translation)}')
print(f'Category KHONG co ban dich: {missing_cats}')
print()
print('=> Ket luan: khong dung FK cung o product_category_name, xu ly bang LEFT JOIN + COALESCE.')

So category trong products: 73
So category co ban dich: 71
Category KHONG co ban dich: {'pc_gamer', 'portateis_cozinha_e_preparadores_de_alimentos'}

=> Ket luan: khong dung FK cung o product_category_name, xu ly bang LEFT JOIN + COALESCE.


## **7. Cardinality Check**
Kiểm tra tỷ lệ 1-N thực tế giữa các bảng — quan trọng để biết JOIN có làm nhân bản dòng dữ liệu
hay không, và để hiểu hành vi khách hàng ở mức tổng quan trước khi vào RFM.

In [12]:
# 1. Ty le customer_id : customer_unique_id — xac nhan 1 nguoi co the co nhieu customer_id
ratio_customers = customers['customer_id'].nunique() / customers['customer_unique_id'].nunique()
print(f'customer_id / customer_unique_id ratio: {ratio_customers:.4f}')
print(f'-> Trung binh moi customer_unique_id co {ratio_customers:.4f} customer_id')
print(f'-> So customer_unique_id xuat hien nhieu hon 1 lan:',
      (customers["customer_unique_id"].value_counts() > 1).sum())

customer_id / customer_unique_id ratio: 1.0348
-> Trung binh moi customer_unique_id co 1.0348 customer_id
-> So customer_unique_id xuat hien nhieu hon 1 lan: 2997


In [13]:
# 2. So don hang trung binh moi khach hang (theo customer_unique_id)
orders_with_unique = orders.merge(customers[['customer_id', 'customer_unique_id']], on='customer_id')
orders_per_customer = orders_with_unique.groupby('customer_unique_id')['order_id'].nunique()

print('So don hang trung binh / khach hang:', round(orders_per_customer.mean(), 3))
print('Median:', orders_per_customer.median())
print('Max:', orders_per_customer.max())
print('% khach hang chi co 1 don:', round((orders_per_customer == 1).mean() * 100, 1), '%')

So don hang trung binh / khach hang: 1.035
Median: 1.0
Max: 17
% khach hang chi co 1 don: 96.9 %


**Nhận xét:** nếu % khách hàng chỉ có 1 đơn rất cao (thường >90% với dataset Olist), điều này
cảnh báo trước: RFM `Frequency` sẽ bị lệch mạnh về giá trị 1 — cần lưu ý khi diễn giải cluster
sau này, đừng ngạc nhiên nếu phần lớn khách hàng rơi vào nhóm Frequency thấp.

In [14]:
# 3. So san pham trung binh moi don hang (order_items : orders)
items_per_order = order_items.groupby('order_id')['order_item_id'].count()
print('So san pham trung binh / don hang:', round(items_per_order.mean(), 3))
print('% don chi co 1 san pham:', round((items_per_order == 1).mean() * 100, 1), '%')

So san pham trung binh / don hang: 1.142
% don chi co 1 san pham: 90.1 %


In [15]:
# 4. So dong payment trung binh moi don hang (co bao nhieu don tra nhieu lan?)
payments_per_order_count = order_payments.groupby('order_id')['payment_sequential'].count()
print('So dong payment trung binh / don hang:', round(payments_per_order_count.mean(), 3))
print('% don co nhieu hon 1 dong payment (VD: voucher + the):',
      round((payments_per_order_count > 1).mean() * 100, 1), '%')

So dong payment trung binh / don hang: 1.045
% don co nhieu hon 1 dong payment (VD: voucher + the): 3.0 %


**Kết luận cardinality:** xác nhận lý do `database/cleaning.sql` phải có bước `payments_per_order`
(SUM theo order_id) trước khi join — nếu không, 1 đơn có nhiều dòng payment sẽ bị nhân bản khi
JOIN trực tiếp với `orders`, làm sai lệch số lượng đơn hàng đếm được.

## **8. Date Range**
Xác định khoảng thời gian dữ liệu bao phủ — quan trọng để chọn `reference_date` hợp lý cho RFM
(`config.ini` → `reference_date = auto` nghĩa là lấy ngày mua gần nhất + 1 ngày).

In [16]:
print('Ngay mua som nhat:', orders['order_purchase_timestamp'].min())
print('Ngay mua muon nhat:', orders['order_purchase_timestamp'].max())
print('Tong so ngay du lieu bao phu:',
      (orders['order_purchase_timestamp'].max() - orders['order_purchase_timestamp'].min()).days, 'ngay')

Ngay mua som nhat: 2016-09-04 21:15:19
Ngay mua muon nhat: 2018-10-17 17:30:18
Tong so ngay du lieu bao phu: 772 ngay


In [17]:
orders_by_year = orders['order_purchase_timestamp'].dt.year.value_counts().sort_index()
print(orders_by_year)
print()
print('=> Neu 1 nam co qua it don (VD: thang dau/cuoi bi cat giua chung),')
print('   can luu y khi dien giai xu huong theo thang o 01_eda.ipynb.')

order_purchase_timestamp
2016      329
2017    45101
2018    54011
Name: count, dtype: int64

=> Neu 1 nam co qua it don (VD: thang dau/cuoi bi cat giua chung),
   can luu y khi dien giai xu huong theo thang o 01_eda.ipynb.


**Nhận xét:** dataset trải dài khoảng 2 năm (2016-2018). Nếu năm đầu/cuối có số đơn thấp bất
thường so với các năm giữa, khả năng cao là do dữ liệu bị cắt giữa chừng (tháng đầu/cuối của
năm đó không có đủ 12 tháng dữ liệu) — cần lưu ý khi so sánh doanh thu giữa các năm, tránh kết
luận sai "năm X giảm doanh thu" trong khi thực chất chỉ là thiếu dữ liệu.

## **9. Data Quality Summary**
Tổng hợp toàn bộ finding từ mục 3-8 thành 1 bảng duy nhất — dùng để tham chiếu nhanh, và là input
trực tiếp cho `docs/data_dictionary.md`.

In [18]:
dq_summary = pd.DataFrame([
    {'Check': 'Missing values', 'Finding': 'orders: null tang dan theo tien trinh giao hang (binh thuong)', 'Action': 'Khong xu ly, la logic nghiep vu'},
    {'Check': 'Missing values', 'Finding': 'products.product_category_name: 610 null', 'Action': 'Giu NULL/unknown khi phan tich category'},
    {'Check': 'Duplicate rows', 'Finding': 'Khong co full-row duplicate o bat ky bang nao', 'Action': 'Khong can xu ly'},
    {'Check': 'Primary Key', 'Finding': 'order_reviews.review_id KHONG unique mot minh (814 trung)', 'Action': 'Dung composite PK (review_id, order_id)'},
    {'Check': 'Foreign Key', 'Finding': 'products.product_category_name co 2 gia tri khong co trong category_translation', 'Action': 'LEFT JOIN + COALESCE, khong dung FK cung'},
    {'Check': 'Cardinality', 'Finding': '1 customer_unique_id co the co nhieu customer_id', 'Action': 'RFM PHAI group theo customer_unique_id'},
    {'Check': 'Cardinality', 'Finding': '1 order co the co nhieu dong payment', 'Action': 'SUM qua payments_per_order truoc khi JOIN'},
    {'Check': 'Date range', 'Finding': 'Du lieu 2016-2018, kiem tra nam dau/cuoi co du 12 thang khong', 'Action': 'Luu y khi so sanh doanh thu theo nam'},
    {'Check': 'order_status', 'Finding': 'delivered chiem ~97% tong don', 'Action': 'Dung lam filter chuan cho RFM'},
])
dq_summary

,Check,Finding,Action
0,Missing values,orders: null tang dan theo tien trinh giao han...,"Khong xu ly, la logic nghiep vu"
1,Missing values,products.product_category_name: 610 null,Giu NULL/unknown khi phan tich category
2,Duplicate rows,Khong co full-row duplicate o bat ky bang nao,Khong can xu ly
3,Primary Key,order_reviews.review_id KHONG unique mot minh ...,"Dung composite PK (review_id, order_id)"
4,Foreign Key,products.product_category_name co 2 gia tri kh...,"LEFT JOIN + COALESCE, khong dung FK cung"
5,Cardinality,1 customer_unique_id co the co nhieu customer_id,RFM PHAI group theo customer_unique_id
6,Cardinality,1 order co the co nhieu dong payment,SUM qua payments_per_order truoc khi JOIN
7,Date range,"Du lieu 2016-2018, kiem tra nam dau/cuoi co du...",Luu y khi so sanh doanh thu theo nam
8,order_status,delivered chiem ~97% tong don,Dung lam filter chuan cho RFM


## **10. Test kết nối PostgreSQL & chạy ingest pipeline**

In [19]:
from src.database import test_connection

assert test_connection(), 'Khong ket noi duoc Postgres — kiem tra .env va docker-compose'

2026-08-04 14:48:00 | INFO     | src.database | Created SQLAlchemy engine for PostgreSQL.
2026-08-04 14:48:00 | INFO     | src.database | Database connection OK.


In [20]:
from src import ingest

ingest.run()

2026-08-04 14:48:00 | INFO     | src.database | Database connection OK.
2026-08-04 14:48:00 | INFO     | src.ingest | Creating schema (schema.sql) ...
2026-08-04 14:48:00 | INFO     | src.database | Executed SQL file: D:\Retail Customer Intelligence Platform\database\schema.sql (26 statements)
2026-08-04 14:48:00 | INFO     | src.ingest | Reading product_category_name_translation.csv ...
2026-08-04 14:48:00 | INFO     | src.ingest | Loading 71 rows into table 'category_translation' ...
2026-08-04 14:48:00 | INFO     | src.ingest | Done: category_translation (71 rows)
2026-08-04 14:48:00 | INFO     | src.ingest | Reading olist_sellers_dataset.csv ...
2026-08-04 14:48:00 | INFO     | src.ingest | Loading 3,095 rows into table 'sellers' ...
2026-08-04 14:48:01 | INFO     | src.ingest | Done: sellers (3,095 rows)
2026-08-04 14:48:01 | INFO     | src.ingest | Reading olist_customers_dataset.csv ...
2026-08-04 14:48:01 | INFO     | src.ingest | Loading 99,441 rows into table 'customers' ...


## **11. Load into PostgreSQL — Verify Row Counts**
So sánh số dòng trong Postgres với số dòng trong CSV gốc — phải khớp 100%.

In [21]:
from src.database import get_engine
from sqlalchemy import text

engine = get_engine()
db_tables = ['customers', 'sellers', 'category_translation', 'products',
             'orders', 'order_items', 'order_payments', 'order_reviews']

with engine.connect() as conn:
    for t in db_tables:
        count = conn.execute(text(f'SELECT COUNT(*) FROM {t}')).scalar()
        expected = len(tables[t])
        status = 'OK' if count == expected else 'LECH SO'
        print(f'{t:25s} Postgres={count:>10,}  CSV={expected:>10,}  -> {status}')

customers                 Postgres=    99,441  CSV=    99,441  -> OK
sellers                   Postgres=     3,095  CSV=     3,095  -> OK
category_translation      Postgres=        71  CSV=        71  -> OK
products                  Postgres=    32,951  CSV=    32,951  -> OK
orders                    Postgres=    99,441  CSV=    99,441  -> OK
order_items               Postgres=   112,650  CSV=   112,650  -> OK
order_payments            Postgres=   103,886  CSV=   103,886  -> OK
order_reviews             Postgres=    99,224  CSV=    99,224  -> OK


## **12. Kết luận thiết kế schema (đã áp dụng vào `database/schema.sql`)**

| Quyết định | Lý do |
|---|---|
| RFM group theo `customer_unique_id`, không phải `customer_id` | `customer_id` là ID theo đơn hàng, không phải theo người (mục 7) |
| Chỉ giữ đơn `order_status = 'delivered'` cho RFM | 97% đơn là delivered, các status khác không đáng tin cậy (mục 3) |
| `products.product_category_name` KHÔNG có FK cứng | Dataset gốc thiếu bản dịch cho vài category (mục 6) |
| `order_reviews` dùng composite PK `(review_id, order_id)` | `review_id` một mình không unique (mục 5) |
| `payments_per_order` phải SUM trước khi JOIN | 1 đơn có thể có nhiều dòng payment (mục 7) |
| `reference_date = auto` trong `config.ini` | Dữ liệu có date range xác định, lấy ngày mua gần nhất + 1 (mục 8) |

**Kết luận chung:** dữ liệu Olist đã được explore đầy đủ ở cả 2 chiều — chất lượng dữ liệu
(missing/duplicate/PK/FK) và cấu trúc quan hệ (cardinality/date range). Các data quality issue
thật đã được phát hiện và xử lý ở tầng schema, không phải patch tạm ở tầng phân tích. Ingest hoàn
tất, sẵn sàng chạy `database/cleaning.sql` + `database/views.sql` (bước tiếp theo).